# NexusChat Internship — Day 4
## Generation Step & Complete RAG Pipeline

**Goal:** Add the generation step (prompt engineering + Gemini call) and wire every module from Days 2, 3, and 4 into one complete, working RAG chatbot.

### What WE will build:
1. **Task 1:** `generator.py` — prompt builder, generation function, display function
2. **Task 2:** `rag_chatbot.py` — full pipeline: index → retrieve → generate
3. **Task 3:** Experiments (remove grounding, vary top_k, add new document)



---
## Setup — Install Dependencies & Imports

In [ ]:
# Install required packages (run once)
!pip install google-generativeai chromadb python-dotenv PyMuPDF python-docx langchain-text-splitters --quiet
print('Dependencies installed.')

In [ ]:
import os
import sys
import json
import google.generativeai as genai
import chromadb
from dotenv import load_dotenv
from IPython.display import display, Markdown

ROOT = os.path.join(os.getcwd(), '..')
ROOT = os.path.abspath(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print(f'Project root: {ROOT}')

from day2.ingestion import load_document
from day2.chunking import chunk_recursive, chunk_fixed, chunk_sentences
from day3.embeddings import embed_text, embed_query, cosine_similarity
print('All modules imported successfully.')

In [ ]:
load_dotenv(os.path.join(ROOT, '.env'))
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

key = os.getenv('GEMINI_API_KEY')
print(f'API key loaded: {key[:8]}...{key[-4:]}' if key else 'ERROR: No API key found')
print(f'Available models: {[m.name for m in genai.list_models()][:5]}...')

---
## Task 1 — The Generation Step



In [ ]:
GENERATION_MODEL = 'models/gemini-3-flash-preview'

def build_prompt(question, retrieved_chunks):
    """
    Constructs the full prompt with system instruction, context block,
    and question. Each chunk is labelled with its source filename.
    """
    context_lines = []
    for chunk in retrieved_chunks:
        source = chunk['source']
        text = chunk['text']
        context_lines.append(f'[Source: {source}]\n{text}')
    context_block = '\n\n'.join(context_lines)

    prompt = f'''You are a helpful assistant that answers questions strictly based on the context documents provided below. Follow these rules:
1. ONLY use information from the CONTEXT section to answer.
2. If the answer is not in the context, say: "I could not find that information in the provided documents."
3. Always end your answer with a SOURCES section that lists the filenames you used.
4. Be concise and direct. Do not add information from outside the context.

--- CONTEXT ---
{context_block}
--- END CONTEXT ---

Question: {question}
Answer:'''
    return prompt


def generate_answer(question, retrieved_chunks):
    prompt = build_prompt(question, retrieved_chunks)
    model = genai.GenerativeModel(GENERATION_MODEL)
    response = model.generate_content(prompt)
    return response.text


def display_answer(question, answer, retrieved_chunks):
    print('\n' + '='*65)
    print(f'QUESTION: {question}')
    print('='*65)
    print(f'\nANSWER:\n{answer}')
    print('\n--- Retrieved chunks passed as context ---')
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f'\nChunk {i} [{chunk["source"]}]:')
        preview = chunk['text'][:150]
        if len(chunk['text']) > 150:
            print(f' {preview}...')
        else:
            print(f' {chunk["text"]}')
    print('='*65)

print('build_prompt, generate_answer, display_answer defined.')

### Test Generation in Isolation (Fake Chunks)



In [ ]:
fake_chunks = [
    {
        'text': 'NexusChat is an enterprise RAG chatbot that answers questions from your documents. It supports PDF, DOCX, and TXT formats.',
        'source': 'sample.pdf',
    },
    {
        'text': 'Users can ask natural language questions and receive cited answers. The system is built on Google Gemini and ChromaDB.',
        'source': 'sample.pdf',
    },
    {
        'text': 'All AI-generated content must be reviewed by a human before publication. Employees must not share confidential data with external AI services.',
        'source': 'sample.docx',
    },
]

print('Fake chunks loaded. Ready to test.')

In [ ]:
q1 = 'What file formats does NexusChat support?'
answer1 = generate_answer(q1, fake_chunks)
display_answer(q1, answer1, fake_chunks)

In [ ]:
q2 = 'What is the pricing of NexusChat?'
answer2 = generate_answer(q2, fake_chunks)
display_answer(q2, answer2, fake_chunks)

---
## Task 2 — Complete RAG Pipeline



In [ ]:
def build_index(doc_paths, collection_name='nexuschat'):
    """
    Loads, chunks, embeds, and stores all documents in ChromaDB.
    Uses in-memory ChromaDB (rebuilds every run).
    """
    print('Building index...')
    client = chromadb.Client()
    collection = client.create_collection(collection_name)

    all_ids, all_embeddings, all_texts, all_meta = [], [], [], []

    for path in doc_paths:
        print(f'  Loading and chunking: {path}')
        text = load_document(path)
        chunks = chunk_recursive(text, size=400, overlap=80)
        for i, chunk in enumerate(chunks):
            chunk_id = f'{os.path.basename(path)}_c{i}'
            embedding = embed_text(chunk)
            all_ids.append(chunk_id)
            all_embeddings.append(embedding)
            all_texts.append(chunk)
            all_meta.append({'source': os.path.basename(path)})

    collection.add(
        ids=all_ids,
        embeddings=all_embeddings,
        documents=all_texts,
        metadatas=all_meta,
    )
    print(f'Index complete -- {len(all_ids)} chunks stored.')
    return collection


def retrieve(collection, question, top_k=3):
    """Embeds the question and returns top_k chunks with source + distance."""
    query_vec = embed_query(question)
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances'],
    )
    chunks = []
    for text, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        chunks.append({
            'text': text,
            'source': meta['source'],
            'distance': dist,
        })
    return chunks


def ask(collection, question, top_k=3):
    """Single-turn Q&A: retrieve + generate + print."""
    chunks = retrieve(collection, question, top_k=top_k)
    answer = generate_answer(question, chunks)
    display_answer(question, answer, chunks)
    return answer, chunks

print('Pipeline functions defined.')

### Build the Index

In [ ]:
base_dir = os.path.join(ROOT, 'day2')
documents = [
    os.path.join(base_dir, 'sample.txt'),
    os.path.join(base_dir, 'sample.pdf'),
    os.path.join(base_dir, 'sample.docx'),
]

for d in documents:
    print(f'  {os.path.exists(d):5} | {d}')

collection = build_index(documents)

### Test Questions

In [ ]:
# Q1: What is NexusChat?
ask(collection, 'What is NexusChat?')

In [ ]:
# Q2: What are the rules about sharing data with AI services?
ask(collection, 'What are the rules about sharing data with AI services?')

In [ ]:
# Q3: How does RAG work?
ask(collection, 'How does RAG work?')

In [ ]:
# Q4: What is the refund policy? (should refuse — not in any document)
ask(collection, 'What is the refund policy?')

In [ ]:
# Q5: Tell me everything you know about NexusChat
ask(collection, 'Tell me everything you know about NexusChat')

---
## Task 3 — Experiments

### Experiment A — Remove the Grounding Instruction



In [ ]:
def build_prompt_weak(question, retrieved_chunks):
    """Same as build_prompt but WITHOUT the grounding instruction (rules 1 & 2 removed)."""
    context_lines = []
    for chunk in retrieved_chunks:
        source = chunk['source']
        text = chunk['text']
        context_lines.append(f'[Source: {source}]\n{text}')
    context_block = '\n\n'.join(context_lines)

    prompt = f'''You are a helpful assistant. Use the context below to answer the question.
--- CONTEXT ---
{context_block}
--- END CONTEXT ---

Question: {question}
Answer:'''
    return prompt


def generate_answer_weak(question, retrieved_chunks):
    prompt = build_prompt_weak(question, retrieved_chunks)
    model = genai.GenerativeModel(GENERATION_MODEL)
    response = model.generate_content(prompt)
    return response.text


print('Weak prompt (no grounding) defined. Asking Q4 again...')
answer_weak = generate_answer_weak('What is the refund policy?', retrieve(collection, 'What is the refund policy?'))
print(f'\nWEAK PROMPT ANSWER:\n{answer_weak}')

### Experiment B — Change top_k

Test with top_k=1 and top_k=6 and compare.

In [ ]:
print('='*65)
print('Experiment B: top_k=1')
print('='*65)
answer_k1, chunks_k1 = ask(collection, 'What is NexusChat?', top_k=1)

print('\n')
print('='*65)
print('Experiment B: top_k=6')
print('='*65)
answer_k6, chunks_k6 = ask(collection, 'What is NexusChat?', top_k=6)

**Observation:** 
- top_k=1: Answer may be too short, missing supporting details.
- top_k=3: Good balance.
- top_k=6: Answer may be repetitive with overlapping chunk content.

### Experiment C — Add a New Document

We add sample2.txt (pricing info) and verify the chatbot can now answer pricing questions.

In [ ]:
# Verify sample2.txt exists
sample2_path = os.path.join(base_dir, 'sample2.txt')
print(f'sample2.txt exists: {os.path.exists(sample2_path)}')
if os.path.exists(sample2_path):
    with open(sample2_path, 'r') as f:
        print(f'Content:\n{f.read()}')

In [ ]:
# Rebuild index WITH sample2.txt
documents_with_pricing = [
    os.path.join(base_dir, 'sample.txt'),
    os.path.join(base_dir, 'sample.pdf'),
    os.path.join(base_dir, 'sample.docx'),
    sample2_path,
]

collection2 = build_index(documents_with_pricing)

In [ ]:
# Ask pricing question — should now answer from sample2.txt
ask(collection2, 'What does the Professional plan cost?')

In [ ]:
# Verify it still refuses questions outside all documents
ask(collection2, 'What is the weather in London?')